# 06 · Автоподбор параметров конвейера v7 по вашей разметке (Optuna)

**Зачем.** У конвейера десятки параметров (размер суперпикселей, пороги слияния, чувствительность к линиям,
ограничения размера/компактности, внешние границы…). Вместо ручного перебора пресетов Optuna (байесовский поиск TPE)
сама пробует наборы параметров на тестовом участке и сравнивает результат с вашей разметкой.

**Как устроена честная проверка.** Разметка делится пополам по долготе: по западной половине (**A**) параметры
подбираются, восточная (**B**) — контроль. Если на B результат тоже вырос — параметры переносятся, а не «подогнаны».

**Что нужно:** тестовый растр (фрагмент вашего композита), разметка полей на него (как для Донбасса),
ноутбук конвейера `field_pipeline_TAT_v7.ipynb` (лежит в `research/classical_v7`; его ячейки исполняются здесь же).
По желанию — внешняя карта границ (`*_ftw_prob.tif` из ноутбука 01, `*_boundary.tif` из 02/03 или
`*_multidate.tif` из 05) в `EXT_BOUNDARY_TIF`: тогда подбирается и её порог.

**Время:** один пробный запуск — 1–3 мин на участке ~20×20 км (CPU). 40 проб ≈ 1–2 ч. Ячейку поиска можно
прерывать и запускать снова — пробы копятся в файле базы Optuna на Диске и не теряются.

### Подключение Google Диска
Выполните ячейку и разрешите доступ к Диску (всплывёт окно Google). Все результаты пишутся в папку
`Colab_Notebooks/Agro_terra/research/…` — её можно поменять в параметрах ниже.

In [ ]:
import os, sys, json, time, glob, shutil, warnings
from pathlib import Path
warnings.filterwarnings('ignore')
LOCAL_TEST = bool(os.environ.get('LOCAL_TEST'))          # True только при проверке вне Colab
if not LOCAL_TEST:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive')
else:
    DRIVE_ROOT = Path(os.environ['LOCAL_DRIVE'])
print('Диск:', DRIVE_ROOT, '| файлов в корне:', len(list(DRIVE_ROOT.iterdir())))

## Шаг 1. Параметры

In [ ]:
PIPELINE_PY = 'Colab_Notebooks/Agro_terra/research/classical_v7/field_pipeline_TAT_v7.ipynb'   # ноутбук v7 (или .py)
TEST_TIF = 'Colab_Notebooks/Agro_terra/test/don_composite.tif'       # тестовый композит (3 канала)
GT_PATH = 'Colab_Notebooks/Agro_terra/test/gt_pieces.gpkg'           # разметка полей на этот участок
GT_TYPE_COL = 'type'
EXT_BOUNDARY_TIF = None              # внешняя карта границ на этот участок (необязательно), напр. '…/don_ftw_prob.tif';
                                     # несколько карт — списком [dict(TIF='…', BAND=1, THR=0.3), …] (тогда пороги не подбираются)
EE_PROJECT = 'my-project-20260614-499410'
BASE_PRESET = 'L_base'               # от какого пресета отталкиваемся
N_TRIALS = 40
OUT_DIR = 'Colab_Notebooks/Agro_terra/research/classical_v7/autotune'
STUDY = 'don_v7'                     # имя «исследования»: продолжить поиск — запустить с тем же именем
WC_NPY = None                        # только для проверок вне Colab

### Функции оценки и картинок (общие для всех ноутбуков)
Ничего менять не нужно — просто выполните ячейку. Метрики те же, что в тестах конвейера
(`tests/eval_gt.py` в репозитории), поэтому числа можно сравнивать с v6:
- **IoU** — для каждого эталонного поля (≥ 5 га) берётся лучший по перекрытию полигон, среднее по полям;
- **IoU≥0.7** — доля эталонных полей, найденных «хорошо»;
- **frag** — сколько полигонов в среднем покрывают одно эталонное поле (1 — идеально, >1 — дробление);
- **merged** — сколько полигонов «склеили» 2 и больше эталонных полей;
- **bF** — точность границ (F-мера, допуск 2 пикселя), **P/R** — точность/полнота линий;
- **охват** — доля площади эталона, покрытая полигонами; по классам (если в эталоне есть колонка `type`).

In [ ]:
import numpy as np, geopandas as gpd, shapely, rasterio, rasterio.windows
from rasterio.features import rasterize
from rasterio.transform import from_origin
from scipy import ndimage as ndi


def load_gt(path, bbox=None):
    """Эталонная разметка: только полигоны в рамке bbox (EPSG:4326: lon_min, lat_min, lon_max, lat_max), без пустых
    и повторных. Так можно указывать разметку на всю область (например DNR_type_2026.geojson), а сравнивать на участке."""
    box = gpd.GeoSeries([shapely.box(*bbox)], crs=4326) if bbox is not None else None
    g = gpd.read_file(path, bbox=box)
    g = g[g.geometry.notna()]
    if bbox is not None:
        g = g[g.to_crs(4326).representative_point().within(box.iloc[0]).values]
    g = g[~g.geometry.to_wkb().duplicated()].reset_index(drop=True)
    print('эталон:', len(g), 'полигонов', '(в рамке участка)' if bbox is not None else '')
    return g


def grid_for(gdf, res=10.0):
    """Сетка 10 м в UTM по охвату слоя (если нет вашего композита)."""
    utm = gdf.estimate_utm_crs()
    x0, y0, x1, y1 = gdf.to_crs(utm).total_bounds
    w, h = int(np.ceil((x1 - x0) / res)) + 2, int(np.ceil((y1 - y0) / res)) + 2
    return from_origin(x0 - res, y1 + res, res, res), (h, w), utm


def evaluate(pred, gt, transform=None, shape=None, crs=None, min_ha=5, type_col='type', label=''):
    """pred, gt — GeoDataFrame полигонов (любая СК). transform/shape/crs — сетка для растровых метрик
    (по умолчанию 10 м UTM по эталону). Печатает строку метрик и возвращает dict."""
    if transform is None:
        transform, shape, crs = grid_for(gt)
    H, W = shape
    gt = gt[gt.geometry.notna()].copy(); pred = pred[pred.geometry.notna()].copy()
    gt = gt[~gt.geometry.to_wkb().duplicated()]                # в выгрузках разметки бывают повторы одного полигона
    box = shapely.box(*rasterio.transform.array_bounds(H, W, transform))
    gt = gt[gt.to_crs(crs).representative_point().within(box).values]   # эталон — только в пределах сетки (участка)
    if not len(gt):
        print(label, ': в пределах участка нет эталонных полей'); return dict(label=label, gt=0)
    win = rasterio.windows.from_bounds(*gt.to_crs(crs).total_bounds, transform)          # сетка — только охват эталона
    win = win.round_offsets().round_lengths()
    c0, r0 = max(int(win.col_off) - 20, 0), max(int(win.row_off) - 20, 0)
    W, H = min(int(win.col_off + win.width) + 20, W) - c0, min(int(win.row_off + win.height) + 20, H) - r0
    transform = rasterio.windows.transform(rasterio.windows.Window(c0, r0, W, H), transform)
    if H * W > 4e8:
        print(f'ВНИМАНИЕ: эталон охватывает {H}×{W} пикселей — оценка займёт много памяти; задайте участок поменьше (BBOX)')
    gt['geometry'] = shapely.make_valid(gt.geometry.values); pred['geometry'] = shapely.make_valid(pred.geometry.values)
    ea = 6933                                                  # равновеликая СК для площадей
    g6 = gt.to_crs(ea); g6 = g6[g6.area / 1e4 >= min_ha].reset_index(drop=True)
    p6 = pred.to_crs(ea); p6['geometry'] = shapely.make_valid(p6.geometry.values)
    tree = shapely.STRtree(p6.geometry.values)
    ious, frag, merged = [], [], np.zeros(len(p6), int)
    for g in g6.geometry.values:
        best, nfr = 0.0, 0
        for j in tree.query(g, predicate='intersects'):
            pj = p6.geometry.values[j]; inter = g.intersection(pj).area
            if inter >= .15 * g.area: merged[j] += 1
            if inter >= .05 * g.area: nfr += 1
            best = max(best, inter / g.union(pj).area)
        ious.append(best); frag.append(nfr)
    ious = np.array(ious)
    idmap = lambda d: rasterize([(g, i + 1) for i, g in enumerate(d.to_crs(crs).geometry.values)],
                                out_shape=(H, W), transform=transform, fill=0, dtype='int32')
    GT, P = idmap(g6), idmap(p6)
    def bnd(m):
        b = np.zeros(m.shape, bool); b[:, :-1] |= m[:, :-1] != m[:, 1:]; b[:-1] |= m[:-1] != m[1:]; return b
    gb, pb = bnd(GT) & (GT > 0), bnd(P) & (P > 0)
    dgt, dpb = ndi.distance_transform_edt(~gb), ndi.distance_transform_edt(~pb)
    dom = ndi.binary_dilation(GT > 0, iterations=3)
    prec = (dgt[pb & dom] <= 2).mean() if (pb & dom).any() else 0.0
    rec = (dpb[gb] <= 2).mean() if gb.any() else 0.0
    bF = 2 * prec * rec / (prec + rec + 1e-9)
    cov = ((P > 0) & (GT > 0)).sum() / max((GT > 0).sum(), 1)
    res = dict(label=label, gt=len(g6), pred=len(pred), IoU=round(float(ious.mean()), 3) if len(ious) else 0,
               IoU07=round(float(np.mean(ious >= .7)), 2) if len(ious) else 0, frag=round(float(np.mean(frag)), 2),
               merged=int((merged >= 2).sum()), bF=round(float(bF), 3), P=round(float(prec), 2), R=round(float(rec), 2),
               cover=round(float(cov), 2))
    if type_col in gt.columns:
        gtt = gt.to_crs(crs)
        for t in sorted(gtt[type_col].dropna().unique()):
            m = rasterize([(g, 1) for g in gtt[gtt[type_col] == t].geometry.values], out_shape=(H, W),
                          transform=transform, fill=0, dtype='uint8') > 0
            res[f'cover_{t}'] = round(float((P[m] > 0).mean()), 2) if m.any() else None
    print(' | '.join(f'{k} {v}' for k, v in res.items()))
    return res


def show_overlay(img_path, layers, title='', bbox=None, out_png=None, bands=(1, 2, 3), figsize=(11, 11)):
    """Картинка: подложка — ваш растр (3 канала, растяжка 2–98 %), поверх контуры слоёв.
    layers — список (GeoDataFrame, цвет, подпись)."""
    import matplotlib.pyplot as plt
    from rasterio.windows import from_bounds
    with rasterio.open(img_path) as s:
        win = from_bounds(*bbox, s.transform) if bbox is not None else rasterio.windows.Window(0, 0, s.width, s.height)
        k = max(1.0, max(win.width, win.height) / 2500)                  # большие охваты читаются с прореживанием
        shp = (len(bands), int(win.height / k), int(win.width / k))
        a = s.read(list(bands), window=win, boundless=True, out_shape=shp).astype(float)
        t = s.window_transform(win) * rasterio.Affine.scale(win.width / shp[2], win.height / shp[1]); crs = s.crs
    for i in range(a.shape[0]):
        lo, hi = np.nanpercentile(a[i][a[i] > 0], [2, 98]) if (a[i] > 0).any() else (0, 1)
        a[i] = np.clip((a[i] - lo) / (hi - lo + 1e-9), 0, 1)
    ext = (t.c, t.c + t.a * a.shape[2], t.f + t.e * a.shape[1], t.f)
    fig, ax = plt.subplots(figsize=figsize)
    ax.imshow(np.moveaxis(a, 0, -1), extent=ext)
    for gdf, color, name in layers:
        if gdf is not None and len(gdf):
            gdf.to_crs(crs).boundary.plot(ax=ax, color=color, linewidth=.8, label=name)
    ax.set_xlim(ext[0], ext[1]); ax.set_ylim(ext[2], ext[3]); ax.set_title(title); ax.legend(loc='upper right')
    if out_png:
        fig.savefig(out_png, dpi=130, bbox_inches='tight')
    plt.show()

### Ядро автоподбора (выполните без изменений)
- `load_pipeline()` — исполняет ячейки конвейера v7 (ноутбук `.ipynb` или скрипт `.py`) в этом же
  ноутбуке: функции этапов становятся доступны, сами этапы не запускаются (включён режим сравнения пресетов).
- `run_trial(params)` — прогоняет один набор параметров (как пресет поверх `BASE_PRESET`) по всем этапам на тестовом
  участке и возвращает финальный слой полей.
- `score(fields)` — метрики на двух половинах разметки: **A** (по ней подбираем) и **B** (контроль — по ней видно,
  не «подогнались» ли параметры под конкретные поля).

In [ ]:
import re, copy, numpy as np, pandas as pd, geopandas as gpd, shapely


def pipeline_cells(path):
    """Ячейки конвейера v7 [(имя, код)] из field_pipeline_TAT_v7.ipynb (имена — в метаданных ячеек;
    если Colab их потерял — узнаём ячейки по содержимому) или из field_pipeline_v7.py (строки `# %% имя`)."""
    text = Path(path).read_text(encoding='utf-8')
    if str(path).endswith('.ipynb'):
        out = []
        for k, c in enumerate(json.loads(text)['cells']):
            if c['cell_type'] != 'code':
                continue
            body = ''.join(c['source']); name = c.get('metadata', {}).get('name')
            if not name:
                name = ('2_params' if re.search(r'^DRIVE_DIR\s*=', body, flags=re.M) else '18_compare' if 'def run_all' in body
                        else '9_segmentation_stage' if 'def get_wc_map' in body else f'cell{k}')
            out.append((name, body))
    else:
        parts = re.split(r'^# %% (\S+)(?: \[markdown\])?\n', text, flags=re.M)
        out = [(parts[i], parts[i + 1]) for i in range(1, len(parts), 2)]
    return [(n, re.sub(r'^\s*!.*$', '', b, flags=re.M)) for n, b in out]      # строки !pip — не Python


def load_pipeline(py_path, overrides, ns):
    """ns — глобальное пространство имён ноутбука (globals()): так параллельные процессы тайлов видят функции конвейера."""
    for name, body in pipeline_cells(py_path):
        if name == '0_intro':
            continue
        if name == '18_compare':                    # нужны только функции (run_all); сам прогон пресетов не запускаем
            c = ns['COMPARE']; ns['COMPARE'] = False
            exec(compile(body, name, 'exec'), ns); ns['COMPARE'] = c
            continue
        exec(compile(body, name, 'exec'), ns)
        if name == '2_params':
            ns.update(overrides); ns.update(COMPARE_PRESETS=True, COMPARE_NAMES=[])
        if name == '9_segmentation_stage' and WC_NPY:                       # только для проверок вне Colab
            _wc = np.load(WC_NPY)
            def _wc_from_npy(t, h, w, tag, _ns=ns):
                c0, r0 = int(round((t.c - _ns['T'].c) / _ns['T'].a)), int(round((t.f - _ns['T'].f) / _ns['T'].e))
                out = np.zeros((h, w), np.uint8); ra, ca = max(r0, 0), max(c0, 0)
                sub = _wc[ra:r0 + h, ca:c0 + w]
                out[ra - r0:ra - r0 + sub.shape[0], ca - c0:ca - c0 + sub.shape[1]] = sub
                return out
            ns['get_wc_map'] = _wc_from_npy
    return ns


def run_trial(ns, params, name='T'):
    p = dict(ns['PRESETS'][BASE_PRESET]); p.update(params)
    ns['PRESETS'][name] = p
    ns['run_all'](name)
    return ns['FIELDS'].copy()


def gt_halves(gt):
    """Делит эталонные поля на западную (A) и восточную (B) половины по медиане долготы центров."""
    c = gt.to_crs(4326).representative_point().x
    m = float(np.median(c))
    return gt[c < m].reset_index(drop=True), gt[c >= m].reset_index(drop=True)


def score(fields, gtA, gtB, grid):
    kw = dict(transform=grid['transform'], shape=grid['shape'], crs=grid['crs'], type_col=GT_TYPE_COL)
    import contextlib, io
    with contextlib.redirect_stdout(io.StringIO()):
        a = evaluate(fields, gtA, label='A', **kw); b = evaluate(fields, gtB, label='B', **kw)
    obj = lambda r: r['IoU'] + 0.5 * r['IoU07'] + 0.5 * r['bF'] - 0.05 * max(r['frag'] - 1.3, 0) - 0.003 * r['merged']
    return obj(a), obj(b), a, b

## Шаг 2. Загружаем конвейер и разметку
Конвейер настраивается на тестовый растр в режиме `test` (весь растр — один тестовый участок), WorldCover — через GEE.

In [ ]:
!pip -q install optuna higra > /dev/null
import optuna, rasterio
OUT = DRIVE_ROOT / OUT_DIR; OUT.mkdir(parents=True, exist_ok=True)
test_tif = DRIVE_ROOT / TEST_TIF
with rasterio.open(test_tif) as s:
    GRID = dict(transform=s.transform, shape=(s.height, s.width), crs=s.crs); TEST_SIZE = max(s.width, s.height) + 10
over = dict(DRIVE_DIR=str(test_tif.parent), COMPOSITE_NAME=test_tif.name, BOUNDARY_NAME=None, OUT_DIR=str(OUT / 'runs'),
            LOCAL_DIR=str(OUT / 'local'), COPY_LOCAL=False, RUN_MODE='test', TEST_SIZE=TEST_SIZE, TEST_CENTER=(0.5, 0.5),
            USE_WORLDCOVER=True, EE_PROJECT=EE_PROJECT)
if WC_NPY:
    from unittest.mock import MagicMock; sys.modules['ee'] = MagicMock()
NS = globals()
load_pipeline(DRIVE_ROOT / PIPELINE_PY, over, NS)
from rasterio.warp import transform_bounds
gt_all = load_gt(DRIVE_ROOT / GT_PATH, transform_bounds(GRID['crs'], 'EPSG:4326', *rasterio.transform.array_bounds(*GRID['shape'], GRID['transform'])))
GT_A, GT_B = gt_halves(gt_all)
print('конвейер загружен; эталон: A', len(GT_A), '| B', len(GT_B))

## Шаг 3. Исходная точка: пресет `BASE_PRESET` как есть (и с внешними границами, если заданы)

In [ ]:
def ext_param():
    if isinstance(EXT_BOUNDARY_TIF, (list, tuple)):
        return [dict(d, TIF=str(DRIVE_ROOT / d['TIF'])) for d in EXT_BOUNDARY_TIF]
    return str(DRIVE_ROOT / EXT_BOUNDARY_TIF)


rows = []
base_params = {} if not EXT_BOUNDARY_TIF else dict(EXT_TIF=ext_param(), EXT_THR=0.3)
for nm, prm in [('base', {})] + ([('base+ext', base_params)] if EXT_BOUNDARY_TIF else []):
    sa, sb, a, b = score(run_trial(NS, prm, 'B0'), GT_A, GT_B, GRID)
    rows.append(dict(name=nm, score_A=round(sa, 3), score_B=round(sb, 3), **{f'A_{k}': v for k, v in a.items() if k != 'label'},
                     **{f'B_{k}': v for k, v in b.items() if k != 'label'}))
BASELINE = pd.DataFrame(rows); display(BASELINE[['name', 'score_A', 'score_B', 'A_IoU', 'B_IoU', 'A_bF', 'B_bF']])

## Шаг 4. Поиск (Optuna TPE)
Подбираются: размер и компактность суперпикселей, способ нарезки (SLIC или Higra), пороги слияния, чувствительность
к линиям и ступенькам, ограничения компактности, сглаживание границ, порог внешней карты (если есть).
Цель = IoU + 0,5·доля(IoU≥0,7) + 0,5·bF − штраф за дробление и склейки (на половине **A**).

In [ ]:
def objective(trial):
    p = dict(SP_SIZE_PX=trial.suggest_int('SP_SIZE_PX', 50, 220, step=10),
             COMPACTNESS=trial.suggest_float('COMPACTNESS', 0.05, 0.8, log=True),
             SEG_METHOD=trial.suggest_categorical('SEG_METHOD', ['slic', 'higra']),
             MERGE_THR=trial.suggest_int('MERGE_THR', 24, 80, step=4),
             JUMP_THR=trial.suggest_int('JUMP_THR', 6, 30, step=2),
             LINE_SENS=trial.suggest_float('LINE_SENS', 0.2, 0.9),
             LINE_EDGE_SENS=trial.suggest_float('LINE_EDGE_SENS', 0.0, 0.6),
             MIN_COMPACT=trial.suggest_float('MIN_COMPACT', 4.0, 9.0),
             RELAX_BETA=trial.suggest_int('RELAX_BETA', 100, 600, step=50))
    if p['SEG_METHOD'] == 'higra':
        p['HG_HIERARCHY'] = trial.suggest_categorical('HG_HIERARCHY', ['area', 'volume', 'dynamics'])
    if EXT_BOUNDARY_TIF:
        p['EXT_TIF'] = ext_param()
        if not isinstance(EXT_BOUNDARY_TIF, (list, tuple)):
            p['EXT_THR'] = trial.suggest_float('EXT_THR', 0.15, 0.8)
    t0 = time.time()
    sa, sb, a, b = score(run_trial(NS, p, 'T'), GT_A, GT_B, GRID)
    for k, v in {**{f'A_{k}': v for k, v in a.items()}, **{f'B_{k}': v for k, v in b.items()}}.items():
        trial.set_user_attr(k, v)
    trial.set_user_attr('score_B', sb)
    print(f'проба {trial.number}: A {sa:.3f} | B {sb:.3f} | IoU A/B {a["IoU"]}/{b["IoU"]} | {time.time() - t0:.0f} с')
    return sa


optuna.logging.set_verbosity(optuna.logging.WARNING)
study = optuna.create_study(study_name=STUDY, direction='maximize', storage=f'sqlite:///{OUT}/optuna.db',
                            load_if_exists=True, sampler=optuna.samplers.TPESampler(seed=1))
left = N_TRIALS - len(study.trials)
if left > 0:
    study.optimize(objective, n_trials=left)
print('лучшая проба:', study.best_trial.number, '| A', round(study.best_value, 3),
      '| B', round(study.best_trial.user_attrs['score_B'], 3))

## Шаг 5. Итоги: лучшие наборы и проверка на половине B
Таблица отсортирована по A; смотрите, чтобы и **B** был выше исходной точки. Лучший набор сохраняется как пресет
(`best_preset.json`) — его можно вставить в `PRESETS` конвейера v7.

In [ ]:
df = study.trials_dataframe(attrs=('number', 'value', 'params', 'user_attrs'))
cols = ['number', 'value', 'user_attrs_score_B', 'user_attrs_A_IoU', 'user_attrs_B_IoU', 'user_attrs_A_bF', 'user_attrs_B_bF',
        'user_attrs_B_IoU07', 'user_attrs_B_frag'] + [c for c in df.columns if c.startswith('params_')]
top = df.sort_values('value', ascending=False)[cols].head(10); display(top); top.to_csv(OUT / 'top_trials.csv', index=False)
display(BASELINE[['name', 'score_A', 'score_B', 'A_IoU', 'B_IoU', 'A_bF', 'B_bF']])
best = dict(study.best_params)
if EXT_BOUNDARY_TIF:
    best['EXT_TIF'] = ext_param()                                  # полный путь /content/drive/MyDrive/…
json.dump({f'TUNED_{STUDY}': {**dict(NS['PRESETS'][BASE_PRESET]), **best}}, open(OUT / 'best_preset.json', 'w'),
          ensure_ascii=False, indent=1, default=str)
print('Сохранено:', OUT / 'best_preset.json')
print(json.dumps(best, ensure_ascii=False, indent=1, default=str))

## Шаг 6. Как применить
1. Откройте `best_preset.json`, скопируйте словарь параметров.
2. В ноутбуке конвейера v7 добавьте его в `PRESETS` под своим именем и сделайте `ACTIVE = '<имя>'`
   (или включите в `COMPARE_NAMES`, чтобы сравнить с `L_base` на других участках).
3. Проверьте на **другом** участке (не тестовом) глазами: параметры подобраны под один участок и одну разметку.